# 07 - Evaluate the LoRA meta-model on the test set

Loads `artifacts/meta_jsonl/test.jsonl`, generates predictions with the LoRA-adapted Gemma-2-9B-it, and reports accuracy / macro-F1 / per-class F1 / confusion matrix.

Also computes three baselines on the same test set for honest comparison:

1. weighted-average argmax (no LLM)
2. best single base model (by val accuracy)
3. zero-shot Gemma-2-9B-it on the same prompts (no LoRA)

In [ ]:
%pip install -q 'transformers>=4.44' 'peft>=0.11' 'bitsandbytes>=0.43' accelerate datasets scikit-learn matplotlib seaborn

In [ ]:
import os, sys, json, re

REPO_ROOT = '/content/drive/MyDrive/thesis/topicmodeling'
TMP_ROOT = '/content/ensemble_tmp'

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)
else:
    import pathlib
    LOCAL = pathlib.Path.cwd().resolve()
    while LOCAL.name != 'tm_research' and LOCAL.parent != LOCAL:
        LOCAL = LOCAL.parent
    sys.path.insert(0, str(LOCAL.parent))

from tm_research.ensemble.colab_setup import setup_colab
paths = setup_colab(repo_root=REPO_ROOT, tmp_root=TMP_ROOT)

import numpy as np, pandas as pd
from tm_research.ensemble.utils_io import (
    META_JSONL_DIR, LORA_DIR, ARTIFACTS_DIR, METRICS_DIR,
    load_label_map, load_probs
)
from tm_research.ensemble.utils_stacking import (
    BASE_MODEL_NAMES, weighted_average, parse_label_from_completion
)
label_map = load_label_map()
with open(ARTIFACTS_DIR / 'weights.json', 'r', encoding='utf-8') as f:
    weights_payload = json.load(f)
weights = weights_payload['weights']
val_acc = weights_payload['val_accuracy_per_model']
print('weights', weights)
print('val accuracies', val_acc)

In [ ]:
test_rows = []
with open(META_JSONL_DIR / 'test.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        test_rows.append(json.loads(line))
y_test = np.array([label_map.label2id[r['gold']] for r in test_rows])
print('test size:', len(test_rows))

## Baselines

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

test_probs = {m: load_probs(m, 'test') for m in BASE_MODEL_NAMES}

def report(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    print(f'\n=== {name} ===\n  acc={acc:.4f}  macro_f1={macro_f1:.4f}  micro_f1={micro_f1:.4f}')
    print(classification_report(
        y_true, y_pred,
        target_names=label_map.class_names,
        zero_division=0,
    ))
    return {'name': name, 'accuracy': acc, 'macro_f1': macro_f1, 'micro_f1': micro_f1}

results = []
weighted_test = weighted_average(test_probs, weights)
results.append(report('baseline_weighted_avg_argmax', y_test, weighted_test.argmax(1)))
best_base = max(val_acc, key=val_acc.get)
results.append(report(f'baseline_best_single_{best_base}', y_test, test_probs[best_base].argmax(1)))

## LoRA meta-model generation

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = 'google/gemma-2-9b-it'
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('Set HF_TOKEN with Gemma-2 license accepted.')

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL_NAME, token=os.environ['HF_TOKEN'])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map='auto',
    torch_dtype=torch.bfloat16, attn_implementation='eager',
    token=os.environ['HF_TOKEN'],
)
lora_meta_path = ARTIFACTS_DIR / 'lora_train_meta.json'
with open(lora_meta_path, 'r', encoding='utf-8') as f:
    lora_meta = json.load(f)
SYSTEM_PROMPT = lora_meta['system_prompt']
model_lora = PeftModel.from_pretrained(base, str(LORA_DIR))
model_lora.eval()

In [ ]:
def build_chat(prompt_block):
    return tok.apply_chat_template(
        [{'role': 'user', 'content': SYSTEM_PROMPT + '\n\n' + prompt_block}],
        tokenize=False, add_generation_prompt=True,
    )

@torch.inference_mode()
def generate_labels(model, rows, batch_size=4, max_new_tokens=12):
    preds = []
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        chats = [build_chat(r['prompt']) for r in batch]
        enc = tok(chats, return_tensors='pt', padding=True, truncation=True, max_length=2048).to(model.device)
        out = model.generate(
            **enc, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0,
            pad_token_id=tok.pad_token_id,
        )
        gen = out[:, enc['input_ids'].shape[1]:]
        decoded = tok.batch_decode(gen, skip_special_tokens=True)
        for txt in decoded:
            lab = parse_label_from_completion(txt, label_map)
            preds.append(lab)
        if i % (batch_size * 10) == 0:
            print(f'  {i}/{len(rows)}')
    return preds

lora_pred_labels = generate_labels(model_lora, test_rows)
fallback = weighted_test.argmax(1)
lora_pred_ids = np.array([
    label_map.label2id[lab] if lab in label_map.label2id else int(fallback[i])
    for i, lab in enumerate(lora_pred_labels)
])
results.append(report('lora_gemma_meta', y_test, lora_pred_ids))

## Zero-shot Gemma baseline (no LoRA, same prompts)

In [ ]:
model_lora = model_lora.unload() if hasattr(model_lora, 'unload') else None
torch.cuda.empty_cache()
zs_pred_labels = generate_labels(base, test_rows)
zs_pred_ids = np.array([
    label_map.label2id[lab] if lab in label_map.label2id else int(fallback[i])
    for i, lab in enumerate(zs_pred_labels)
])
results.append(report('zero_shot_gemma', y_test, zs_pred_ids))

## Confusion matrix for the LoRA meta-model

In [ ]:
cm = confusion_matrix(y_test, lora_pred_ids, labels=list(range(label_map.num_classes)))
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=label_map.class_names, yticklabels=label_map.class_names,
)
plt.xlabel('Predicted'); plt.ylabel('Gold'); plt.title('LoRA-Gemma meta-model (test)')
plt.tight_layout()
from pathlib import Path
out_dir = Path(REPO_ROOT) / 'Confusion_matrix'
out_dir.mkdir(parents=True, exist_ok=True)
out_png = out_dir / 'ensemble_llm_meta.png'
plt.savefig(out_png, dpi=150)
plt.show()
print('saved', out_png)

In [ ]:
summary_path = METRICS_DIR / 'ensemble_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print('saved', summary_path)
pd.DataFrame(results)

In [ ]:
from tm_research.ensemble.utils_io import push_artifacts_to_persistent
push_artifacts_to_persistent()